In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sqlalchemy.exc import OperationalError

load_dotenv()

db = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
port = os.getenv("POSTGRES_PORT", "5432")
host = os.getenv("POSTGRES_HOST", "127.0.0.1")

df = pd.DataFrame()
try:
    engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")
    df = pd.read_sql("SELECT * FROM jobs", engine)
    if df.empty:
        print("In database there are no vacancies!")
    else:
        print(f"In database there are {len(df)} vacancies!")
        df["date_collected"] = pd.to_datetime(df["date_collected"])
        df.set_index("date_collected", inplace=True)
        df.sort_index(inplace=True)
        df.drop(columns=["id"], inplace=True, errors="ignore")
        print(df.head())
except (OperationalError, Exception) as error:
    print(error)


def get_skills_counter(date_target, level_target):
    if df.empty:
        return Counter()
    try:
        df_by_date = df.loc[date_target]
        if level_target != "all_levels":
            df_by_date = df_by_date[df_by_date["level"] == level_target]

        counts = Counter()
        for s in df_by_date["skills"]:
            counts.update(s)
        return counts
    except KeyError:
        return Counter()


In [ ]:
# Find a level "junior", "middle", "senior" or "all_levels"
level = "middle"
# Find a date "2026-03-26"
date = "2026-04-01"
# Find top skills
top_skills = 25

total_skills = get_skills_counter(date, level)
if not total_skills:
    print(f"No data found for level '{level}' on date {date}.")
else:
    skills_series = pd.Series(total_skills).sort_values(ascending=False).head(top_skills)

    skills_series.plot(
        kind="bar",
        color="blue",
        figsize=(15, 7),
        width=0.7
    )
    plt.title(f"Top {top_skills} skills for {date} on {level} in Python.")
    plt.show()


In [ ]:
# Find a level "junior", "middle", "senior" or "all_levels"
level = "middle"
# Find a date_one and date_two "2026-03-26 "
date_one = "2026-03-31"
date_two = "2026-04-01"
# Find top skills
top = 15

skills_one = get_skills_counter(date_one, level)
skills_two = get_skills_counter(date_two, level)
if not skills_one:
    print(f"No data found for level '{level}' on date {date_one}.")
elif not skills_two:
    print(f"No data found for level '{level}' on date {date_two}.")
else:
    df_comparison = pd.DataFrame({
        date_one: skills_one,
        date_two: skills_two,
    }).fillna(0).sort_values(by=date_two, ascending=False).head(top)

    df_comparison.plot(
        kind="bar",
        color=["blue", "green"],
        figsize=(15, 7),
        width=0.7
    )
    plt.title(f"Comparison of skills: for {date_one} vs {date_two} (level: '{level}') in Python.")
    plt.ylabel("Count")
    plt.legend(title="Date")
    plt.show()